In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [76]:
import pandas as pd

# Load the CSV file
df = pd.read_csv('/content/drive/MyDrive/Sentiment_Analysis/imdb_combined.csv')

# Display the first five rows
df.head()

,review,sentiment
0,"This film is fun, if your a person who likes a...",pos
1,My fondness for Chris Rock varies with his mov...,pos
2,High school friends Andre Kriegman and Cal Gab...,pos
3,The story is about a little girl growing up in...,pos
4,"Then you must see this film, to understand the...",pos


In [77]:
# Check duplicate records
print("Duplicate records before cleaning:", df.duplicated().sum())

# Remove exact duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

print("Duplicate records after cleaning:", df.duplicated().sum())
print("Dataset shape after removing duplicates:", df.shape)

Duplicate records before cleaning: 428
Duplicate records after cleaning: 0
Dataset shape after removing duplicates: (49582, 2)


In [78]:
# Check whether the same review has different sentiment labels

conflicting_labels = (
    df.groupby('review')['sentiment']
      .nunique()
)

conflicting_labels = conflicting_labels[
    conflicting_labels > 1
]

print(
    "Reviews with conflicting sentiment labels:",
    len(conflicting_labels)
)

Reviews with conflicting sentiment labels: 0


In [79]:
import re

# Common contractions
contractions = {
    "can't": "can not",
    "cannot": "can not",
    "couldn't": "could not",
    "didn't": "did not",
    "doesn't": "does not",
    "don't": "do not",
    "hadn't": "had not",
    "hasn't": "has not",
    "haven't": "have not",
    "isn't": "is not",
    "it's": "it is",
    "wasn't": "was not",
    "weren't": "were not",
    "won't": "will not",
    "wouldn't": "would not",
    "shouldn't": "should not",
    "I've": "I have",
    "I'll": "I will",
    "I'd": "I would",
    "I'm": "I am",
    "you're": "you are",
    "they're": "they are",
    "we're": "we are"
}


def clean_text(text):

    # Convert to string
    text = str(text)

    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', ' ', text)

    # Expand contractions
    for contraction, expansion in contractions.items():
        text = re.sub(
            r'\b' + re.escape(contraction) + r'\b',
            expansion,
            text,
            flags=re.IGNORECASE
        )

    # Convert to lowercase
    text = text.lower()

    # Keep only letters and spaces
    text = re.sub(r'[^a-z\s]', ' ', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()


    return text

In [80]:
sample = """
I REALLY enjoyed this movie! <br /><br />
It was amazing. Visit www.example.com
"""

print("Before:")
print(sample)

print("\nAfter:")
print(clean_text(sample))

Before:

I REALLY enjoyed this movie! <br /><br />
It was amazing. Visit www.example.com


After:
i really enjoyed this movie it was amazing visit


In [81]:
# Apply the cleaning function
df['clean_review'] = df['review'].apply(clean_text)

# Display original and cleaned reviews
display(
    df[['review', 'clean_review', 'sentiment']].head(10)
)

,review,clean_review,sentiment
0,"This film is fun, if your a person who likes a...",this film is fun if your a person who likes a ...,pos
1,My fondness for Chris Rock varies with his mov...,my fondness for chris rock varies with his mov...,pos
2,High school friends Andre Kriegman and Cal Gab...,high school friends andre kriegman and cal gab...,pos
3,The story is about a little girl growing up in...,the story is about a little girl growing up in...,pos
4,"Then you must see this film, to understand the...",then you must see this film to understand the ...,pos
5,Go see this movie for the gorgeous imagery of ...,go see this movie for the gorgeous imagery of ...,pos
6,Scream was Wes Craven's last decent thriller. ...,scream was wes craven s last decent thriller s...,pos
7,This is the best movie I've come across in a l...,this is the best movie i have come across in a...,pos
8,A superb and compelling drama about the hunt f...,a superb and compelling drama about the hunt f...,pos
9,If you enjoy Cleese & all the British 'Pythone...,if you enjoy cleese all the british pythonesqu...,pos


In [82]:
import nltk
from nltk.corpus import stopwords

# Download the stopwords corpus
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Create a set of all unique words (tokens) present in the cleaned reviews
all_tokens = set(" ".join(df['clean_review'].fillna("")).split())

# Unique stop words present in df
unique_stops = sorted(all_tokens & stop_words)

print("Count:", len(unique_stops))
print(unique_stops)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Count: 153
['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', 'd', 'did', 'didn', 'do', 'does', 'doesn', 'doing', 'don', 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', 'has', 'hasn', 'have', 'haven', 'having', 'he', 'her', 'here', 'hers', 'herself', 'him', 'himself', 'his', 'how', 'i', 'if', 'in', 'into', 'is', 'isn', 'it', 'its', 'itself', 'just', 'll', 'm', 'ma', 'me', 'mightn', 'more', 'most', 'mustn', 'my', 'myself', 'needn', 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', 'she', 'should', 'shouldn', 'so', 'some', 'such', 't', 'than', 'that', 'the', 'their', 'theirs', 'them', 'themselves', 'then', 'there', 'these', 'they', 'this', 'those', 'through', 'to', 'too', 'under', 'until

In [83]:
# Stop words to KEEP (they affect sentiment)
keep_words = {
    # negations
    "no", "nor", "not", "don", "didn", "doesn", "isn", "wasn", "weren", "aren",
    "hasn", "haven", "hadn", "couldn", "wouldn", "shouldn", "won", "mustn",
    "needn", "shan", "mightn", "ain",
    # intensifiers / degree
    "very", "too", "more", "most", "just", "so", "only", "few", "again",
    # contrast
    "but", "against", "off", "down", "up", "over", "under",
}

# Stop words to REMOVE
custom_stop_words = set(stopwords.words("english")) - keep_words

def remove_stopwords(text):
    return " ".join(w for w in str(text).split() if w not in custom_stop_words)

df["clean_review_nostop"] = df["clean_review"].fillna("").apply(remove_stopwords)

df[["clean_review", "clean_review_nostop"]].head()

,clean_review,clean_review_nostop
0,this film is fun if your a person who likes a ...,film fun person likes good campy feature film ...
1,my fondness for chris rock varies with his mov...,fondness chris rock varies movies hated lethal...
2,high school friends andre kriegman and cal gab...,high school friends andre kriegman cal gabriel...
3,the story is about a little girl growing up in...,story little girl growing up colonial africa b...
4,then you must see this film to understand the ...,must see film understand reality read book ms ...


In [84]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

def stem_text(text):
    return " ".join(stemmer.stem(word) for word in str(text).split())

df["clean_review_stemmed"] = (
    df["clean_review_nostop"]
    .fillna("")
    .apply(stem_text)
)

display(
    df[["clean_review_nostop", "clean_review_stemmed"]].head(10)
)

,clean_review_nostop,clean_review_stemmed
0,film fun person likes good campy feature film ...,film fun person like good campi featur film ev...
1,fondness chris rock varies movies hated lethal...,fond chri rock vari movi hate lethal weapon bu...
2,high school friends andre kriegman cal gabriel...,high school friend andr kriegman cal gabriel d...
3,story little girl growing up colonial africa b...,stori littl girl grow up coloni africa but so ...
4,must see film understand reality read book ms ...,must see film understand realiti read book ms ...
5,go see movie gorgeous imagery andy goldsworthy...,go see movi gorgeou imageri andi goldsworthi s...
6,scream wes craven last decent thriller since n...,scream we craven last decent thriller sinc not...
7,best movie come across long not only best movi...,best movi come across long not onli best movi ...
8,superb compelling drama hunt notorious russian...,superb compel drama hunt notori russian serial...
9,enjoy cleese british pythonesque humour time l...,enjoy clees british pythonesqu humour time lit...


In [85]:
for i in range(5):
    print("ORIGINAL:")
    print(df["clean_review_nostop"].iloc[i])
    print("\nSTEMMED:")
    print(df["clean_review_stemmed"].iloc[i])
    print("\n" + "="*100 + "\n")

ORIGINAL:
film fun person likes good campy feature film every no means movie fine cinema but dont take things too seriously laugh elvira good frownbuster

STEMMED:
film fun person like good campi featur film everi no mean movi fine cinema but dont take thing too serious laugh elvira good frownbust


ORIGINAL:
fondness chris rock varies movies hated lethal weapon but hated everyone movie like not holding back like dogma well best yet not expecting good laughed arse off whole time chris rock delivers sweet wonderful story backed funniest comedy seen quite time loved

STEMMED:
fond chri rock vari movi hate lethal weapon but hate everyon movi like not hold back like dogma well best yet not expect good laugh ars off whole time chri rock deliv sweet wonder stori back funniest comedi seen quit time love


ORIGINAL:
high school friends andre kriegman cal gabriel declare war classmates plan terrifying assault high school begin deadly countdown final act revenge two start video diary explain fee

In [86]:
import nltk
from nltk.stem import WordNetLemmatizer

nltk.download("wordnet")
nltk.download("omw-1.4")

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    return " ".join(
        lemmatizer.lemmatize(word)
        for word in str(text).split()
    )

df["clean_review_lemmatized"] = (
    df["clean_review_nostop"]
    .fillna("")
    .apply(lemmatize_text)
)

display(
    df[["clean_review_nostop", "clean_review_lemmatized"]].head(10)
)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,clean_review_nostop,clean_review_lemmatized
0,film fun person likes good campy feature film ...,film fun person like good campy feature film e...
1,fondness chris rock varies movies hated lethal...,fondness chris rock varies movie hated lethal ...
2,high school friends andre kriegman cal gabriel...,high school friend andre kriegman cal gabriel ...
3,story little girl growing up colonial africa b...,story little girl growing up colonial africa b...
4,must see film understand reality read book ms ...,must see film understand reality read book m d...
5,go see movie gorgeous imagery andy goldsworthy...,go see movie gorgeous imagery andy goldsworthy...
6,scream wes craven last decent thriller since n...,scream wes craven last decent thriller since n...
7,best movie come across long not only best movi...,best movie come across long not only best movi...
8,superb compelling drama hunt notorious russian...,superb compelling drama hunt notorious russian...
9,enjoy cleese british pythonesque humour time l...,enjoy cleese british pythonesque humour time l...


In [87]:
for i in range(5):
    print("ORIGINAL:")
    print(df["clean_review_nostop"].iloc[i])
    print("\nSTEMMED:")
    print(df["clean_review_lemmatized"].iloc[i])
    print("\n" + "="*100 + "\n")

ORIGINAL:
film fun person likes good campy feature film every no means movie fine cinema but dont take things too seriously laugh elvira good frownbuster

STEMMED:
film fun person like good campy feature film every no mean movie fine cinema but dont take thing too seriously laugh elvira good frownbuster


ORIGINAL:
fondness chris rock varies movies hated lethal weapon but hated everyone movie like not holding back like dogma well best yet not expecting good laughed arse off whole time chris rock delivers sweet wonderful story backed funniest comedy seen quite time loved

STEMMED:
fondness chris rock varies movie hated lethal weapon but hated everyone movie like not holding back like dogma well best yet not expecting good laughed arse off whole time chris rock delivers sweet wonderful story backed funniest comedy seen quite time loved


ORIGINAL:
high school friends andre kriegman cal gabriel declare war classmates plan terrifying assault high school begin deadly countdown final act rev

In [88]:
df.head()

,review,sentiment,clean_review,clean_review_nostop,clean_review_stemmed,clean_review_lemmatized
0,"This film is fun, if your a person who likes a...",pos,this film is fun if your a person who likes a ...,film fun person likes good campy feature film ...,film fun person like good campi featur film ev...,film fun person like good campy feature film e...
1,My fondness for Chris Rock varies with his mov...,pos,my fondness for chris rock varies with his mov...,fondness chris rock varies movies hated lethal...,fond chri rock vari movi hate lethal weapon bu...,fondness chris rock varies movie hated lethal ...
2,High school friends Andre Kriegman and Cal Gab...,pos,high school friends andre kriegman and cal gab...,high school friends andre kriegman cal gabriel...,high school friend andr kriegman cal gabriel d...,high school friend andre kriegman cal gabriel ...
3,The story is about a little girl growing up in...,pos,the story is about a little girl growing up in...,story little girl growing up colonial africa b...,stori littl girl grow up coloni africa but so ...,story little girl growing up colonial africa b...
4,"Then you must see this film, to understand the...",pos,then you must see this film to understand the ...,must see film understand reality read book ms ...,must see film understand realiti read book ms ...,must see film understand reality read book m d...


In [89]:
df = df[["clean_review_lemmatized", "sentiment"]]

df = df.rename(columns={
    "clean_review_lemmatized": "Review",
})


In [90]:
output_path = "/content/drive/MyDrive/Sentiment_Analysis/imdb_cleaned.csv"
df.to_csv(output_path, index=False)

print(f"CSV file saved at: {output_path}")


CSV file saved at: /content/drive/MyDrive/Sentiment_Analysis/imdb_cleaned.csv
